<a href="https://colab.research.google.com/github/fideitosrcr/proyect_2/blob/master/Proyecto_versi%C3%B3n_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pyreadr

In [ ]:
import zipfile
import os

zip_path = "/content/biotime_v2.zip"
extract_to = "/content/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("Archivos descomprimidos:")
for root, dirs, files in os.walk("/content/biotime_v2"):
    for f in files:
        print(os.path.join(root, f))

Archivos descomprimidos:
/content/biotime_v2/logs/bronze_20260528_020118.log
/content/biotime_v2/logs/bronze_20260528_021815.log
/content/biotime_v2/logs/pipeline_20260528_024528.log
/content/biotime_v2/logs/bronze_20260528_021057.log
/content/biotime_v2/logs/bronze_20260528_014051.log
/content/biotime_v2/logs/pipeline_20260528_023509.log
/content/biotime_v2/logs/bronze_20260528_021510.log
/content/biotime_v2/logs/bronze_20260528_021306.log
/content/biotime_v2/logs/pipeline_20260528_023720.log
/content/biotime_v2/logs/bronze_20260528_014049.log
/content/biotime_v2/bronze/data_contract_bronze.json
/content/biotime_v2/bronze/bronze_profile.json
/content/biotime_v2/bronze/biotime_bronze_v2.duckdb
/content/biotime_v2/raw/biotime_v2_query_15April25.rds
/content/biotime_v2/raw/biotime_v2_metadata_15April25.csv
/content/biotime_v2/notebooks/02_silver.ipynb
/content/biotime_v2/notebooks/03_gold.ipynb
/content/biotime_v2/notebooks/05_report_generator.ipynb
/content/biotime_v2/notebooks/01_bronz

In [ ]:




import subprocess, sys, pkgutil

if not pkgutil.find_loader("pyreadr"):
    print("Instalando pyreadr...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyreadr"])


import os
import hashlib
import json
import logging
import duckdb
import pyreadr
import pandas as pd
from datetime import datetime

BASE_DIR = "/content/biotime_v2"
RAW_DIR = os.path.join(BASE_DIR, "raw")
BRONZE_DIR = os.path.join(BASE_DIR, "bronze")
SILVER_DIR = os.path.join(BASE_DIR, "silver")
GOLD_DIR = os.path.join(BASE_DIR, "gold")
LOG_DIR = os.path.join(BASE_DIR, "logs")

for d in [RAW_DIR, BRONZE_DIR, SILVER_DIR, GOLD_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# Logging
log_file = os.path.join(LOG_DIR, f"pipeline_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler(log_file), logging.StreamHandler(sys.stdout)]
)


# VERIFICAR ARCHIVOS RAW

RDS_PATH = os.path.join(RAW_DIR, "biotime_v2_query_15April25.rds")
CSV_PATH = os.path.join(RAW_DIR, "biotime_v2_metadata_15April25.csv")

for path in [RDS_PATH, CSV_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"No se encuentra {path}. Sube los archivos a /content/biotime_v2/raw/")


# BRONZE

logging.info("=" * 50)
logging.info("INICIANDO CAPA BRONZE")
logging.info("=" * 50)

def sha256_file(filepath):
    hash_obj = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            hash_obj.update(chunk)
    return hash_obj.hexdigest()

def profile_dataframe(df, name):
    return {
        "name": name, "rows": len(df), "columns": df.shape[1],
        "missing_values": df.isnull().sum().to_dict(),
        "memory_usage_mb": df.memory_usage(deep=True).sum() / (1024**2)
    }

logging.info("Leyendo RDS...")
df_data = pyreadr.read_r(RDS_PATH)[None]
logging.info(f"Datos: {df_data.shape}")

logging.info("Leyendo CSV...")
df_meta = pd.read_csv(CSV_PATH)
logging.info(f"Metadatos: {df_meta.shape}")

profile_data = profile_dataframe(df_data, "biotime_data")
profile_meta = profile_dataframe(df_meta, "biotime_metadata")

CONTRACT = {
    "dataset": "biotime", "version": "2.0.0",
    "expected_columns_data": ["ID_ALL_RAW_DATA", "ABUNDANCE", "YEAR", "STUDY_ID", "valid_name"],
    "expected_columns_meta": ["STUDY_ID", "REALM", "HABITAT", "CLIMATE"],
    "quality_checks": [{"name": "no_null_study_id", "threshold": 0}, {"name": "year_range", "params": {"min": 1800, "max": 2025}, "threshold": 0}]
}

missing_data = set(CONTRACT["expected_columns_data"]) - set(df_data.columns)
missing_meta = set(CONTRACT["expected_columns_meta"]) - set(df_meta.columns)
if missing_data or missing_meta:
    raise ValueError(f"Columnas faltantes: datos {missing_data}, metadatos {missing_meta}")

hash_raw = sha256_file(RDS_PATH)
logging.info(f"Hash: {hash_raw}")

DB_PATH = os.path.join(BRONZE_DIR, "biotime_bronze_v2.duckdb")
conn = duckdb.connect(DB_PATH)
conn.execute("CREATE SCHEMA IF NOT EXISTS bronze")
conn.execute("CREATE SCHEMA IF NOT EXISTS meta")

conn.execute("""
    CREATE TABLE IF NOT EXISTS meta.ingestion_log (
        run_id INTEGER PRIMARY KEY, timestamp_utc TIMESTAMP, source_file_hash VARCHAR,
        n_rows INTEGER, n_columns INTEGER, status VARCHAR, error_message VARCHAR
    )
""")
conn.execute("DELETE FROM meta.ingestion_log WHERE run_id = 1")
conn.execute("""
    INSERT INTO meta.ingestion_log (run_id, timestamp_utc, source_file_hash, n_rows, n_columns, status)
    VALUES (1, CURRENT_TIMESTAMP, ?, ?, ?, 'STARTED')
""", [hash_raw, len(df_data), len(df_data.columns)])

conn.execute("""
    CREATE OR REPLACE TABLE bronze.biotime_data AS
    SELECT *, EXTRACT(YEAR FROM CURRENT_TIMESTAMP) AS ingestion_year FROM df_data
""")
conn.register("df_meta", df_meta)
conn.execute("CREATE OR REPLACE TABLE bronze.biotime_metadata AS SELECT * FROM df_meta")

null_study = conn.execute("SELECT COUNT(*) FROM bronze.biotime_data WHERE STUDY_ID IS NULL").fetchone()[0]
invalid_years = conn.execute("SELECT COUNT(*) FROM bronze.biotime_data WHERE YEAR < 1800 OR YEAR > 2025").fetchone()[0]

status = "FAIL" if null_study > 0 or invalid_years > 0 else "PASS"
error_msg = None
if null_study > 0: error_msg = f"null_study_id = {null_study}"
elif invalid_years > 0: error_msg = f"invalid_years = {invalid_years}"

conn.execute("UPDATE meta.ingestion_log SET status = ?, error_message = ? WHERE run_id = 1", [status, error_msg])
if status == "FAIL": raise RuntimeError(error_msg)

# Guardar evidencias
with open(os.path.join(BRONZE_DIR, "data_contract_bronze.json"), "w") as f:
    json.dump(CONTRACT, f, indent=2)
with open(os.path.join(BRONZE_DIR, "bronze_profile.json"), "w") as f:
    json.dump({
        "data_profile": profile_data, "meta_profile": profile_meta,
        "hash_raw": hash_raw, "validation": {"null_study_id": null_study, "invalid_years": invalid_years}
    }, f, indent=2)

logging.info("Bronze completado.")
conn.close()


# SILVER

logging.info("=" * 50)
logging.info("INICIANDO CAPA SILVER")
logging.info("=" * 50)

conn = duckdb.connect(DB_PATH)
total_rows = conn.execute("SELECT COUNT(*) FROM bronze.biotime_data").fetchone()[0]
logging.info(f"Filas en bronze: {total_rows}")

profile_before = conn.execute("""
    SELECT COUNT(*) AS total, COUNT(DISTINCT STUDY_ID) AS unique_studies,
           COUNT(DISTINCT valid_name) AS unique_species,
           SUM(CASE WHEN ABUNDANCE <= 0 OR ABUNDANCE IS NULL THEN 1 ELSE 0 END) AS invalid_abundance,
           SUM(CASE WHEN YEAR < 1800 OR YEAR > 2025 THEN 1 ELSE 0 END) AS invalid_years,
           SUM(CASE WHEN valid_name IS NULL THEN 1 ELSE 0 END) AS null_species
    FROM bronze.biotime_data
""").df()

CONFIG = {"max_abundance": 1e6, "min_year": 1800, "max_year": 2025, "valid_latitude_range": (-90, 90), "valid_longitude_range": (-180, 180)}

silver_sql = f"""
    CREATE OR REPLACE TABLE silver_cleaned AS
    WITH base AS (
        SELECT STUDY_ID, ID_ALL_RAW_DATA, YEAR, MONTH, DAY,
            CASE WHEN ABUNDANCE > {CONFIG['max_abundance']} THEN NULL ELSE ABUNDANCE END AS abundance_clean,
            COALESCE(LOWER(TRIM(valid_name)), 'unknown') AS species_canon,
            COALESCE(LOWER(TRIM(taxon)), 'unknown') AS taxon_group,
            LATITUDE, LONGITUDE, DEPTH,
            ROW_NUMBER() OVER (PARTITION BY ID_ALL_RAW_DATA ORDER BY YEAR) AS rn
        FROM bronze.biotime_data WHERE valid_name IS NOT NULL
    ),
    quality_flags AS (
        SELECT *,
            (YEAR BETWEEN {CONFIG['min_year']} AND {CONFIG['max_year']}) AS is_year_valid,
            (abundance_clean > 0 AND abundance_clean IS NOT NULL) AS is_abundance_valid,
            (LATITUDE BETWEEN {CONFIG['valid_latitude_range'][0]} AND {CONFIG['valid_latitude_range'][1]}) AS is_latitude_valid,
            (LONGITUDE BETWEEN {CONFIG['valid_longitude_range'][0]} AND {CONFIG['valid_longitude_range'][1]}) AS is_longitude_valid
        FROM base WHERE rn = 1
    )
    SELECT STUDY_ID, ID_ALL_RAW_DATA, YEAR, MONTH, DAY, abundance_clean AS abundance,
           species_canon, taxon_group, LATITUDE, LONGITUDE, DEPTH,
           is_year_valid, is_abundance_valid, is_latitude_valid, is_longitude_valid,
           CURRENT_TIMESTAMP AS silver_ts
    FROM quality_flags
"""
conn.execute(silver_sql)
silver_rows = conn.execute("SELECT COUNT(*) FROM silver_cleaned").fetchone()[0]
logging.info(f"Silver: {silver_rows} filas (eliminadas {total_rows - silver_rows})")

profile_after = conn.execute("""
    SELECT COUNT(*) AS total, COUNT(DISTINCT STUDY_ID) AS unique_studies, COUNT(DISTINCT species_canon) AS unique_species,
           SUM(CASE WHEN is_abundance_valid = FALSE THEN 1 ELSE 0 END) AS invalid_abundance,
           SUM(CASE WHEN is_year_valid = FALSE THEN 1 ELSE 0 END) AS invalid_years,
           SUM(CASE WHEN species_canon = 'unknown' THEN 1 ELSE 0 END) AS unknown_species
    FROM silver_cleaned
""").df()

conn.execute("""
    CREATE OR REPLACE TABLE meta.quality_events AS
    SELECT CURRENT_TIMESTAMP AS ts_utc, 'null_species' AS check_name, 'critical' AS severity,
        (SELECT COUNT(*) FROM silver_cleaned WHERE species_canon = 'unknown') AS metric_value, 'especies sin nombre taxonómico' AS details
    UNION ALL SELECT CURRENT_TIMESTAMP, 'invalid_year', 'high',
        (SELECT COUNT(*) FROM silver_cleaned WHERE is_year_valid = FALSE), 'años fuera de rango 1800-2025'
    UNION ALL SELECT CURRENT_TIMESTAMP, 'invalid_abundance', 'medium',
        (SELECT COUNT(*) FROM silver_cleaned WHERE is_abundance_valid = FALSE), 'abundancia nula, cero o >1e6'
    UNION ALL SELECT CURRENT_TIMESTAMP, 'invalid_coordinates', 'medium',
        (SELECT COUNT(*) FROM silver_cleaned WHERE is_latitude_valid = FALSE OR is_longitude_valid = FALSE), 'latitud/longitud fuera de rango'
""")

conn.execute("ALTER TABLE silver_cleaned ADD COLUMN decade INTEGER")
conn.execute("UPDATE silver_cleaned SET decade = (YEAR // 10) * 10")

csv_dir = os.path.join(SILVER_DIR, "silver_biotime_cleaned")
os.makedirs(csv_dir, exist_ok=True)
conn.execute(f"COPY silver_cleaned TO '{csv_dir}/silver_data.csv' (FORMAT CSV, HEADER)")
logging.info(f"CSV exportado a {csv_dir}/silver_data.csv")

quality_df = conn.execute("SELECT * FROM meta.quality_events").df()
with open(os.path.join(SILVER_DIR, "silver_evidence.json"), "w") as f:
    json.dump({
        "config": CONFIG, "profile_before": profile_before.to_dict(),
        "profile_after": profile_after.to_dict(), "rows_removed": total_rows - silver_rows,
        "quality_events": quality_df.to_dict(orient='records')
    }, f, indent=2, default=str)

logging.info("Silver completado.")
conn.close()


# GOLD

logging.info("=" * 50)
logging.info("INICIANDO CAPA GOLD")
logging.info("=" * 50)

conn = duckdb.connect()
silver_path = os.path.join(SILVER_DIR, "silver_biotime_cleaned", "**/*.csv")
conn.execute(f"CREATE OR REPLACE VIEW silver_view AS SELECT * FROM read_csv_auto('{silver_path}')")
total_rows = conn.execute("SELECT COUNT(*) FROM silver_view").fetchone()[0]
logging.info(f"Filas cargadas desde Silver: {total_rows}")

bronze_db = os.path.join(BRONZE_DIR, "biotime_bronze_v2.duckdb")
conn.execute(f"ATTACH '{bronze_db}' AS bronze_db (READ_ONLY)")

# ARREGLO: Crear schema gold explícitamente
conn.execute("CREATE SCHEMA IF NOT EXISTS gold")

conn.execute("""
    CREATE OR REPLACE TABLE gold.dim_study AS
    SELECT s.STUDY_ID, ANY_VALUE(m.REALM) AS realm, ANY_VALUE(m.HABITAT) AS habitat,
           ANY_VALUE(m.CLIMATE) AS climate, MIN(s.YEAR) AS first_record_year, MAX(s.YEAR) AS last_record_year
    FROM silver_view s LEFT JOIN bronze_db.bronze.biotime_metadata m ON s.STUDY_ID = m.STUDY_ID GROUP BY s.STUDY_ID
""")
logging.info("dim_study creada.")

conn.execute("""
    CREATE OR REPLACE TABLE gold.dim_species AS
    SELECT ROW_NUMBER() OVER (ORDER BY species_canon) AS species_id, species_canon AS species_name, taxon_group
    FROM (SELECT DISTINCT species_canon, taxon_group FROM silver_view) s
""")
logging.info("dim_species creada.")

conn.execute("""
    CREATE OR REPLACE TABLE gold.dim_time AS
    SELECT DISTINCT YEAR, (YEAR // 10) * 10 AS decade, (YEAR >= 2000) AS is_2000s FROM silver_view ORDER BY YEAR
""")
logging.info("dim_time creada.")

conn.execute("""
    CREATE OR REPLACE TABLE gold.fact_observation AS
    SELECT ROW_NUMBER() OVER () AS observation_id, s.STUDY_ID, sp.species_id, s.YEAR, s.abundance,
           s.LATITUDE, s.LONGITUDE, s.DEPTH, s.is_abundance_valid, s.is_year_valid,
           s.is_latitude_valid, s.is_longitude_valid, s.silver_ts
    FROM silver_view s JOIN gold.dim_species sp ON s.species_canon = sp.species_name
""")
logging.info("fact_observation creada.")

conn.execute("""
    CREATE OR REPLACE TABLE gold.study_habitat_bridge AS
    SELECT DISTINCT STUDY_ID, HABITAT FROM bronze_db.bronze.biotime_metadata WHERE HABITAT IS NOT NULL
""")
logging.info("study_habitat_bridge creada.")

conn.execute("CREATE INDEX IF NOT EXISTS idx_fact_study ON gold.fact_observation (STUDY_ID)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_fact_species ON gold.fact_observation (species_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_species_name ON gold.dim_species (species_name)")
logging.info("Índices creados.")

conn.execute("""
    CREATE OR REPLACE TABLE gold.richness_by_habitat AS
    SELECT h.habitat, COUNT(DISTINCT f.species_id) AS total_species, COUNT(*) AS total_observations, AVG(f.abundance) AS avg_abundance
    FROM gold.fact_observation f JOIN gold.study_habitat_bridge h ON f.STUDY_ID = h.STUDY_ID GROUP BY h.habitat ORDER BY total_species DESC
""")

conn.execute("""
    CREATE OR REPLACE TABLE gold.richness_trend AS
    SELECT t.decade, COUNT(DISTINCT f.species_id) AS species_count, COUNT(*) AS observation_count, AVG(f.abundance) AS avg_abundance
    FROM gold.fact_observation f JOIN gold.dim_time t ON f.YEAR = t.YEAR GROUP BY t.decade ORDER BY t.decade
""")

conn.execute("""
    CREATE OR REPLACE TABLE gold.top_species AS
    SELECT sp.species_name, SUM(f.abundance) AS total_abundance, COUNT(*) AS occurrence_count, AVG(f.abundance) AS mean_abundance
    FROM gold.fact_observation f JOIN gold.dim_species sp ON f.species_id = sp.species_id
    WHERE f.is_abundance_valid GROUP BY sp.species_name ORDER BY total_abundance DESC LIMIT 20
""")
logging.info("Vistas materializadas creadas.")

tables = ["dim_study", "dim_species", "dim_time", "fact_observation",
          "study_habitat_bridge", "richness_by_habitat", "richness_trend", "top_species"]
for table in tables:
    out_path = os.path.join(GOLD_DIR, f"{table}.parquet")
    conn.execute(f"COPY gold.{table} TO '{out_path}' (FORMAT PARQUET)")
    logging.info(f"Exportada {table} a {out_path}")

orphans = conn.execute("""
    SELECT COUNT(*) FROM gold.fact_observation f LEFT JOIN gold.dim_study s ON f.STUDY_ID = s.STUDY_ID WHERE s.STUDY_ID IS NULL
""").fetchone()[0]
logging.info(f"Integridad referencial: {'OK' if orphans == 0 else f'{orphans} huérfanos'}")

explain_result = conn.execute("""
    EXPLAIN ANALYZE SELECT h.habitat, COUNT(DISTINCT f.species_id) FROM gold.fact_observation f
    JOIN gold.study_habitat_bridge h ON f.STUDY_ID = h.STUDY_ID GROUP BY h.habitat
""").df()
with open(os.path.join(GOLD_DIR, "explain_analyze_richness_by_habitat.txt"), "w") as f:
    f.write(str(explain_result))

conn.close()
logging.info("Gold finalizado.")
logging.info("=" * 50)
logging.info("PIPELINE COMPLETO EJECUTADO CON ÉXITO")
logging.info("=" * 50)

/tmp/ipykernel_19444/2235123721.py:11: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if not pkgutil.find_loader("pyreadr"):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:


import pandas as pd

# Leer los Parquet generados y mostrarlos
print("\n" + "="*60)
print("TABLAS GENERADAS EN GOLD")
print("="*60)

# Dimensión estudios
dim_study = pd.read_parquet(os.path.join(GOLD_DIR, "dim_study.parquet"))
print(f"\n📊 DIM_STUDY ({len(dim_study)} filas):")
print(dim_study.head(10).to_string())

# Dimensión especies
dim_species = pd.read_parquet(os.path.join(GOLD_DIR, "dim_species.parquet"))
print(f"\n📊 DIM_SPECIES ({len(dim_species)} filas):")
print(dim_species.head(10).to_string())

# Tabla de hechos
fact_obs = pd.read_parquet(os.path.join(GOLD_DIR, "fact_observation.parquet"))
print(f"\n📊 FACT_OBSERVATION ({len(fact_obs)} filas):")
print(fact_obs.head(5).to_string())

# Riqueza por hábitat
richness_habitat = pd.read_parquet(os.path.join(GOLD_DIR, "richness_by_habitat.parquet"))
print(f"\n📊 RIQUEZA POR HÁBITAT ({len(richness_habitat)} filas):")
print(richness_habitat.to_string())

# Tendencia temporal
richness_trend = pd.read_parquet(os.path.join(GOLD_DIR, "richness_trend.parquet"))
print(f"\n📊 TENDENCIA TEMPORAL ({len(richness_trend)} filas):")
print(richness_trend.to_string())

# Top especies
top_species = pd.read_parquet(os.path.join(GOLD_DIR, "top_species.parquet"))
print(f"\n📊 TOP ESPECIES ({len(top_species)} filas):")
print(top_species.head(10).to_string())


TABLAS GENERADAS EN GOLD

📊 DIM_STUDY (708 filas):
   STUDY_ID        realm                     habitat             climate  first_record_year  last_record_year
0        33       Marine                Seaweed beds           Temperate               1992              2009
1       201  Terrestrial             Tropical forest            Tropical               1987              2000
2       202  Terrestrial             Tropical forest            Tropical               1988              2000
3        87       Marine                Seaweed beds           Temperate               1982              1990
4       120       Marine              oceanic waters  Temperate/Tropical               2002              2008
5        59  Terrestrial              Urban / Desert           Temperate               1977              2002
6       171       Marine    Multiple marine habitats  Temperate/Tropical               1988              2008
7        53  Terrestrial  Savanna/ Tallgrass prairie           Tempe

In [ ]:



import os
import json
import base64
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from IPython.display import HTML, display

# RUTAS LOCALES
BASE_DIR = "/content/biotime_v2"
REPORT_DIR = os.path.join(BASE_DIR, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)


# FUNCIONES SEGURAS CON MANEJO DE ERRORES


def safe_load_json(path):
    """Carga JSON con manejo de errores detallado."""
    if not os.path.exists(path):
        print(f"⚠️ No existe: {path}")
        return {}
    try:
        with open(path, 'r') as f:
            data = json.load(f)
        # Verificar que sea dict
        if not isinstance(data, dict):
            print(f"⚠️ {path} no es un diccionario válido. Tipo: {type(data)}")
            return {}
        return data
    except json.JSONDecodeError as e:
        print(f"❌ Error JSON en {path}: {e}")
        return {}
    except Exception as e:
        print(f"❌ Error leyendo {path}: {e}")
        return {}

def safe_load_parquet(path):
    """Carga Parquet con verificación."""
    if not os.path.exists(path):
        print(f"⚠️ No existe: {path}")
        return pd.DataFrame()
    try:
        df = pd.read_parquet(path)
        print(f"✅ Cargado: {path} ({len(df)} filas)")
        return df
    except Exception as e:
        print(f"❌ Error Parquet {path}: {e}")
        return pd.DataFrame()

def safe_get(data, *keys, default='N/A'):
    """Navega diccionarios anidados de forma segura."""
    try:
        for key in keys:
            if isinstance(data, dict):
                data = data.get(key, default)
            elif isinstance(data, list) and isinstance(key, int):
                data = data[key] if key < len(data) else default
            else:
                return default
        return data if data is not None else default
    except Exception:
        return default

def img_to_base64(path):
    """Convierte imagen a base64."""
    if not os.path.exists(path):
        return ""
    try:
        with open(path, "rb") as f:
            return base64.b64encode(f.read()).decode()
    except Exception as e:
        print(f"⚠️ Error imagen {path}: {e}")
        return ""

def safe_to_html(df, max_rows=10):
    """Convierte DataFrame a HTML de forma segura."""
    if df is None or df.empty:
        return "<p><em>Sin datos disponibles.</em></p>"
    try:
        return df.head(max_rows).to_html(index=False, classes='data-table')
    except Exception as e:
        print(f"⚠️ Error to_html: {e}")
        return "<p><em>Error al generar tabla.</em></p>"


# CARGAR EVIDENCIAS


print("=" * 60)
print("CARGANDO EVIDENCIAS")
print("=" * 60)

bronze_profile = safe_load_json(os.path.join(BASE_DIR, "bronze/bronze_profile.json"))
silver_evidence = safe_load_json(os.path.join(BASE_DIR, "silver/silver_evidence.json"))

# Debug: mostrar qué se cargó
print(f"\nBronze keys: {list(bronze_profile.keys()) if bronze_profile else 'VACÍO'}")
print(f"Silver keys: {list(silver_evidence.keys()) if silver_evidence else 'VACÍO'}")


# CARGAR TABLAS GOLD


print("\n" + "=" * 60)
print("CARGANDO TABLAS GOLD")
print("=" * 60)

richness_habitat = safe_load_parquet(os.path.join(BASE_DIR, "gold/richness_by_habitat.parquet"))
richness_trend = safe_load_parquet(os.path.join(BASE_DIR, "gold/richness_trend.parquet"))
top_species = safe_load_parquet(os.path.join(BASE_DIR, "gold/top_species.parquet"))


# PROCESAR QUALITY EVENTS


quality_events = pd.DataFrame()
if silver_evidence and "quality_events" in silver_evidence:
    events = silver_evidence["quality_events"]
    print(f"\nTipo de quality_events: {type(events)}")

    try:
        if isinstance(events, list):
            quality_events = pd.DataFrame(events)
        elif isinstance(events, dict):
            # Si es dict de listas (formato orient='records')
            quality_events = pd.DataFrame(events)
        else:
            print(f"⚠️ Formato inesperado de quality_events: {type(events)}")
    except Exception as e:
        print(f"❌ Error creando DataFrame quality_events: {e}")

print(f"Quality events: {len(quality_events)} filas")


# CREAR GRÁFICOS


print("\n" + "=" * 60)
print("GENERANDO GRÁFICOS")
print("=" * 60)

sns.set_style("whitegrid")

# Gráfico 1: Riqueza por hábitat
fig1, ax1 = plt.subplots(figsize=(10, 6))
if not richness_habitat.empty and 'total_species' in richness_habitat.columns and 'habitat' in richness_habitat.columns:
    try:
        plot_data = richness_habitat.head(10).sort_values('total_species', ascending=False)
        sns.barplot(data=plot_data, x='total_species', y='habitat', palette='viridis', ax=ax1)
        ax1.set_title('Top 10 Hábitats por Riqueza de Especies', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Total Especies')
        ax1.set_ylabel('Hábitat')
    except Exception as e:
        print(f"⚠️ Error gráfico 1: {e}")
        ax1.text(0.5, 0.5, f'Error: {e}', ha='center', va='center', transform=ax1.transAxes)
else:
    ax1.text(0.5, 0.5, 'Sin datos de riqueza por hábitat', ha='center', va='center', transform=ax1.transAxes)
    ax1.set_title('Top 10 Hábitats por Riqueza de Especies', fontsize=14)

plt.tight_layout()
fig1_path = os.path.join(REPORT_DIR, "richness_habitat.png")
plt.savefig(fig1_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ Gráfico 1 guardado: {fig1_path}")

# Gráfico 2: Tendencia temporal
fig2, ax2 = plt.subplots(figsize=(10, 6))
if not richness_trend.empty and 'decade' in richness_trend.columns and 'species_count' in richness_trend.columns:
    try:
        ax2.plot(richness_trend['decade'], richness_trend['species_count'],
                 marker='o', linewidth=3, markersize=10, color='#2ecc71')
        ax2.fill_between(richness_trend['decade'], richness_trend['species_count'],
                         alpha=0.3, color='#2ecc71')
        ax2.set_title('Evolución de Riqueza por Década', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Década')
        ax2.set_ylabel('Número de Especies')
        ax2.grid(True, alpha=0.3)
    except Exception as e:
        print(f"⚠️ Error gráfico 2: {e}")
        ax2.text(0.5, 0.5, f'Error: {e}', ha='center', va='center', transform=ax2.transAxes)
else:
    ax2.text(0.5, 0.5, 'Sin datos de tendencia temporal', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('Evolución de Riqueza por Década', fontsize=14)

plt.tight_layout()
fig2_path = os.path.join(REPORT_DIR, "richness_trend.png")
plt.savefig(fig2_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ Gráfico 2 guardado: {fig2_path}")

# Gráfico 3: Top especies
fig3, ax3 = plt.subplots(figsize=(10, 6))
if not top_species.empty and 'total_abundance' in top_species.columns and 'species_name' in top_species.columns:
    try:
        plot_data = top_species.head(10).sort_values('total_abundance', ascending=False)
        sns.barplot(data=plot_data, x='total_abundance', y='species_name', palette='magma', ax=ax3)
        ax3.set_title('Top 10 Especies más Abundantes', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Abundancia Total')
        ax3.set_ylabel('Especie')
    except Exception as e:
        print(f"⚠️ Error gráfico 3: {e}")
        ax3.text(0.5, 0.5, f'Error: {e}', ha='center', va='center', transform=ax3.transAxes)
else:
    ax3.text(0.5, 0.5, 'Sin datos de especies', ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('Top 10 Especies más Abundantes', fontsize=14)

plt.tight_layout()
fig3_path = os.path.join(REPORT_DIR, "top_species.png")
plt.savefig(fig3_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ Gráfico 3 guardado: {fig3_path}")


# CONVERTIR IMÁGENES A BASE64


img1_b64 = img_to_base64(fig1_path)
img2_b64 = img_to_base64(fig2_path)
img3_b64 = img_to_base64(fig3_path)

print(f"\nImágenes base64: {len(img1_b64)} | {len(img2_b64)} | {len(img3_b64)} caracteres")


# PREPARAR DATOS PARA HTML


print("\n" + "=" * 60)
print("PREPARANDO DATOS DEL REPORTE")
print("=" * 60)

# Bronze metrics (usando safe_get para navegar anidado)
bronze_rows = safe_get(bronze_profile, 'data_profile', 'rows', default='N/A')
bronze_meta_rows = safe_get(bronze_profile, 'meta_profile', 'rows', default='N/A')
bronze_hash = safe_get(bronze_profile, 'hash_raw', default='N/A')
null_study = safe_get(bronze_profile, 'validation', 'null_study_id', default='N/A')
invalid_years = safe_get(bronze_profile, 'validation', 'invalid_years', default='N/A')

# Silver metrics
silver_before = safe_get(silver_evidence, 'profile_before', 'total', 0, default='N/A')
if isinstance(silver_before, list):
    silver_before = silver_before[0] if len(silver_before) > 0 else 'N/A'

silver_after = safe_get(silver_evidence, 'profile_after', 'total', 0, default='N/A')
if isinstance(silver_after, list):
    silver_after = silver_after[0] if len(silver_after) > 0 else 'N/A'

silver_removed = safe_get(silver_evidence, 'rows_removed', default='N/A')

print(f"Bronze rows: {bronze_rows}, Silver before: {silver_before}, after: {silver_after}")

# Tablas HTML
quality_html = safe_to_html(quality_events)
top_species_html = safe_to_html(top_species, max_rows=10)


# GENERAR HTML


print("\n" + "=" * 60)
print("GENERANDO HTML")
print("=" * 60)

html_content = f"""<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Reporte BioTIME v2.0</title>
    <style>
        :root {{
            --primary: #2c3e50; --secondary: #3498db; --accent: #16a085;
            --danger: #e74c3c; --warning: #f39c12; --light: #ecf0f1;
        }}
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            margin: 0; padding: 0;
            background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
            color: var(--primary); line-height: 1.6;
        }}
        .container {{ max-width: 1200px; margin: 0 auto; padding: 40px 20px; }}
        header {{
            background: var(--primary); color: white; padding: 30px;
            border-radius: 15px; margin-bottom: 30px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.2);
        }}
        header h1 {{ margin: 0; font-size: 2.5em; font-weight: 300; }}
        header p {{ margin: 10px 0 0 0; opacity: 0.9; font-size: 1.1em; }}
        .section {{
            background: white; padding: 30px; margin-bottom: 25px;
            border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.08);
        }}
        .section h2 {{
            color: var(--secondary); border-bottom: 3px solid var(--secondary);
            padding-bottom: 10px; margin-top: 0;
        }}
        .section h3 {{ color: var(--accent); margin-top: 25px; }}
        .metric-box {{
            background: linear-gradient(135deg, var(--accent) 0%, #1abc9c 100%);
            color: white; padding: 20px; border-radius: 10px; margin: 15px 0;
        }}
        .metric-box .label {{
            font-size: 0.9em; opacity: 0.9; text-transform: uppercase; letter-spacing: 1px;
        }}
        .metric-box .value {{ font-size: 2em; font-weight: bold; margin-top: 5px; }}
        .grid {{
            display: grid; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
            gap: 20px; margin: 20px 0;
        }}
        table {{
            width: 100%; border-collapse: collapse; margin: 15px 0;
            font-size: 0.95em; box-shadow: 0 2px 8px rgba(0,0,0,0.05);
        }}
        th {{
            background: var(--secondary); color: white; padding: 14px;
            text-align: left; font-weight: 600;
        }}
        td {{ padding: 12px 14px; border-bottom: 1px solid #e0e0e0; }}
        tr:hover {{ background: #f8f9fa; }}
        tr:nth-child(even) {{ background: #fafbfc; }}
        img {{
            max-width: 100%; height: auto; border-radius: 8px;
            margin: 20px 0; box-shadow: 0 4px 12px rgba(0,0,0,0.15);
        }}
        .chart-container {{
            background: #fafbfc; padding: 20px; border-radius: 10px; margin: 20px 0;
        }}
        .footer {{
            text-align: center; padding: 30px; color: #7f8c8d;
            font-size: 0.9em; border-top: 2px solid var(--light); margin-top: 40px;
        }}
        .highlight {{
            background: #fff3cd; border-left: 4px solid var(--warning);
            padding: 15px; margin: 15px 0; border-radius: 0 8px 8px 0;
        }}
        .success {{
            background: #d4edda; border-left: 4px solid var(--accent);
        }}
        .data-table {{ font-size: 0.9em; }}
    </style>
</head>
<body>
    <div class="container">
        <header>
            <h1>🌿 BioTIME v2.0</h1>
            <p>Reporte de Ingeniería de Datos | Generado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        </header>

        <div class="section">
            <h2>1. Resumen Ejecutivo</h2>
            <div class="grid">
                <div class="metric-box">
                    <div class="label">Datos Crudos</div>
                    <div class="value">{bronze_rows}</div>
                </div>
                <div class="metric-box">
                    <div class="label">Metadatos</div>
                    <div class="value">{bronze_meta_rows}</div>
                </div>
                <div class="metric-box">
                    <div class="label">Filas Limpias</div>
                    <div class="value">{silver_after}</div>
                </div>
                <div class="metric-box">
                    <div class="label">Eliminadas</div>
                    <div class="value">{silver_removed}</div>
                </div>
            </div>
            <div class="highlight success">
                <strong>Hash SHA-256:</strong> {bronze_hash}<br>
                <strong>Validaciones:</strong> Nulos STUDY_ID = {null_study} | Años inválidos = {invalid_years}
            </div>
        </div>

        <div class="section">
            <h2>2. Calidad de Datos (Silver)</h2>
            <p><strong>Filas antes:</strong> {silver_before} | <strong>Después:</strong> {silver_after} | <strong>Removidas:</strong> {silver_removed}</p>
            <h3>Eventos de Calidad</h3>
            {quality_html}
        </div>

        <div class="section">
            <h2>3. Análisis Gold - Modelo Estrella</h2>

            <div class="chart-container">
                <h3>Top 10 Hábitats por Riqueza de Especies</h3>
                <img src="data:image/png;base64,{img1_b64}" alt="Riqueza por hábitat">
            </div>

            <div class="chart-container">
                <h3>Evolución Temporal de la Riqueza</h3>
                <img src="data:image/png;base64,{img2_b64}" alt="Tendencia temporal">
            </div>

            <div class="chart-container">
                <h3>Top 10 Especies más Abundantes</h3>
                <img src="data:image/png;base64,{img3_b64}" alt="Top especies">
            </div>

            <h3>Tabla de Top Especies</h3>
            {top_species_html}
        </div>

        <div class="section">
            <h2>4. Conclusiones y Recomendaciones</h2>
            <ul>
                <li>✅ Pipeline ejecutado con trazabilidad completa (hash SHA-256).</li>
                <li>✅ Reglas de calidad detectaron y mitigaron valores atípicos en abundancia y coordenadas.</li>
                <li>✅ Modelo estrella implementado con claves surrogate y tabla puente many-to-many.</li>
                <li>✅ Índices ART creados para optimización de joins en DuckDB.</li>
                <li>⚠️ Se recomienda actualizar metadatos periódicamente para mejorar integridad referencial.</li>
                <li>⚠️ Considerar particionamiento adicional por región geográfica para datasets futuros.</li>
            </ul>
        </div>

        <div class="footer">
            <p>🔬 <strong>BioTIME v2.0</strong> | Pipeline de Ingeniería de Datos</p>
            <p>Tecnologías: Python + DuckDB + Pandas + Matplotlib</p>
            <p>Generado automáticamente el {datetime.now().strftime('%d/%m/%Y a las %H:%M')}</p>
        </div>
    </div>
</body>
</html>
"""


# GUARDAR Y MOSTRAR RESULTADO


report_path = os.path.join(REPORT_DIR, "biotime_report.html")
try:
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(html_content)
    print(f"\n{'='*60}")
    print(f"✅ REPORTE GENERADO EXITOSAMENTE")
    print(f"{'='*60}")
    print(f"📄 HTML: {report_path}")
    print(f"📊 Tamaño: {len(html_content):,} caracteres")
    print(f"\nPara descargar:")
    print(f"   1. Panel archivos (📁) → biotime_v2/reports/")
    print(f"   2. Descarga 'biotime_report.html'")

    display(HTML(f"""
    <div style="background:#d4edda;padding:15px;border-radius:8px;margin:10px 0;">
    <h3>✅ Reporte generado correctamente</h3>
    <p><b>Ubicación:</b> {report_path}</p>
    <p><b>Tamaño:</b> {len(html_content):,} caracteres</p>
    </div>
    """))

except Exception as e:
    print(f"❌ Error guardando reporte: {e}")

CARGANDO EVIDENCIAS

Bronze keys: ['data_profile', 'meta_profile', 'hash_raw', 'validation']
Silver keys: ['config', 'profile_before', 'profile_after', 'rows_removed', 'quality_events']

CARGANDO TABLAS GOLD
✅ Cargado: /content/biotime_v2/gold/richness_by_habitat.parquet (326 filas)
✅ Cargado: /content/biotime_v2/gold/richness_trend.parquet (16 filas)
✅ Cargado: /content/biotime_v2/gold/top_species.parquet (20 filas)

Tipo de quality_events: <class 'list'>
Quality events: 4 filas

GENERANDO GRÁFICOS
✅ Gráfico 1 guardado: /content/biotime_v2/reports/richness_habitat.png
✅ Gráfico 2 guardado: /content/biotime_v2/reports/richness_trend.png


/tmp/ipykernel_19444/3600383886.py:205: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=plot_data, x='total_abundance', y='species_name', palette='magma', ax=ax3)


✅ Gráfico 3 guardado: /content/biotime_v2/reports/top_species.png

Imágenes base64: 44468 | 97052 | 82804 caracteres

PREPARANDO DATOS DEL REPORTE
Bronze rows: 11989233, Silver before: N/A, after: N/A

GENERANDO HTML

✅ REPORTE GENERADO EXITOSAMENTE
📄 HTML: /content/biotime_v2/reports/biotime_report.html
📊 Tamaño: 233,459 caracteres

Para descargar:
   1. Panel archivos (📁) → biotime_v2/reports/
   2. Descarga 'biotime_report.html'


In [ ]:
import os

base = "/content/biotime_v2"
print(f"¿Existe {base}? {os.path.exists(base)}")

if os.path.exists(base):
    print("\n📁 Archivos en biotime_v2:")
    for item in os.listdir(base):
        full_path = os.path.join(base, item)
        if os.path.isfile(full_path):
            print(f"   📄 {item}")
        elif os.path.isdir(full_path):
            print(f"   📂 {item}/")
            # Mostrar contenido de subcarpetas relevantes
            if item in ['bronze', 'silver', 'gold', 'raw', 'reports', 'logs']:
                try:
                    subitems = os.listdir(full_path)[:5]  # primeros 5
                    for sub in subitems:
                        print(f"      - {sub}")
                except:
                    pass
else:
    print("❌ La carpeta NO EXISTE")

¿Existe /content/biotime_v2? True

📁 Archivos en biotime_v2:
   📂 logs/
      - orchestrator_20260528_030940.log
      - pipeline_20260528_024941.log
      - bronze_20260528_020118.log
      - orchestrator_20260528_030511.log
      - bronze_20260528_021815.log
   📂 .ipynb_checkpoints/
   📂 gold/
      - richness_by_habitat.parquet
      - dim_study.parquet
      - dim_species.parquet
      - fact_observation.parquet
      - study_habitat_bridge.parquet
   📂 bronze/
      - data_contract_bronze.json
      - bronze_profile.json
      - biotime_bronze_v2.duckdb
   📂 reports/
      - richness_trend.png
      - top_species.png
      - biotime_report.html
      - richness_habitat.png
   📂 raw/
      - biotime_v2_query_15April25.rds
      - biotime_v2_metadata_15April25.csv
   📂 notebooks/
   📂 silver/
      - silver_evidence.json
      - silver_biotime_cleaned
